# 🧹 Limpieza de datos

Notebook interactivo para explorar, limpiar y exportar los datos.
Al final, los datos limpios se guardan en `data/processed/datos_limpios.csv`
para ser consumidos por el dashboard.

In [2]:
# Ajuste de ruta para importar desde src/
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import re 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_CSV, PROCESSED_CSV, RAW_DIR, PROCESSED_DIR

# Configuración de visualización
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print(f"CSV crudo: {RAW_CSV}")
print(f"CSV limpio: {PROCESSED_CSV}")

CSV crudo: C:\Users\Angel\Desktop\Sistemas Inteligentes\LimpiezaData\data\raw\datos_raw.csv
CSV limpio: C:\Users\Angel\Desktop\Sistemas Inteligentes\LimpiezaData\data\processed\datos_limpios.csv


## 1. Cargar datos crudos

In [13]:
df = pd.read_csv(RAW_CSV)
print(f"Shape original: {df.shape}")
df.head()

Shape original: (20, 21)


,Marca temporal,Dirección de correo electrónico,¿Deseas participar en el estudio?,¿A qué área de conocimiento pertenece tu carrera?,¿Cuál es tu carrera?,¿En qué semestre te encuentras actualmente?,¿Tienes pareja actualmente?,¿A qué edad fue tu primera relación amorosa?,¿Qué significa para ti estar enamorado?,¿Cuál consideras que es la base de la confianza?,¿Cómo defines el respeto dentro de una relación?,¿Qué cualidad valoras más en una pareja y por qué?,Describe físicamente a tu pareja ideal,"¿Cuáles son tus ""no negociables"" en una relación?",¿Cuáles son las red flags (señales de alerta) que consideras dentro de una relación?,¿Cuál es la diferencia de edad máxima que aceptarías en una pareja?,¿Cómo demuestras cariño sin usar palabras?,¿Qué es lo más significativo que alguien ha hecho por ti?,¿Cuál sería para ti la forma correcta de resolver un desacuerdo con tu pareja?,"Describe brevemente una cita ideal, sin importar presupuesto ni lugar.","¿Qué canción, película o libro describe mejor tu idea del amor?"
0,12/10/2026 16:26:08,321184074@pcpuma.acatlan.unam.mx,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30/10/2026 18:40:27,321188065@pcpuma.acatlan.unam.mx,Sí,Área 1: Ciencias Físico-Matemáticas y de las I...,Matemáticas Aplicadas y Computación,7.0,Si,14.0,Es un sentimiento dirigido hacia otra persona ...,Honestidad y confianza,No lastimar física o emocionalmente a la otra ...,"El apoyo, porque me hace sentir especial e imp...",No me fijo físicamente en alguien.,"Infidelidad, mentiras, engaños y tratar de con...",Frases para controlar o atacar emocionalmente ...,+/- 2 años,Querer estar cerca de la otra persona.\nDarle ...,Tratarme cómo una persona normal y ser paciente.,Hablar de lo que pasó cuando el enojo haya pas...,Un viaje a otro país.,Cómo si fuera la primera vez
2,30/10/2026 18:55:47,424059594@pcpuma.acatlan.unam.mx,Sí,Área 1: Ciencias Físico-Matemáticas y de las I...,Matemáticas Aplicadas y Computación,7.0,No,12.0,Que una persona te genere muchas emociones y n...,Honestidad y lealtad,Ser leal,Cariño ya que me hace sentir querido por esa p...,"Leal, respetuosa, cariñosa y con buen sentido ...",Infidelidad y Falta de Respeto,Que me hable mucho de sus relaciones anteriore...,+/- 4 años,"Tiempo, detalles, palabras y sobre todo estar ...",Escucharme hablar del mismo tema infinidad de ...,Hablando y teniendo una buena comunicación,Un día en un parque de diversiones y después u...,La canción Mi Riqueza
3,30/10/2026 23:58:24,423085822@pcpuma.acatlan.unam.mx,Sí,Área 1: Ciencias Físico-Matemáticas y de las I...,MAC,7.0,Si,15.0,Escuche Amar y querer de José José para más de...,"Comunicación, la capacidad de ser vulnerable.","No pasarse de ""chistoso"".","Comunicación, porque así puedo estar en sus za...",De baja estatura.,-Infidelidad. \n-Manipulación.\n-Falta de empa...,"El que violente física, emocionalmente de mi. ...",+/- 3 años,"Tener retroalimentación, pasar un rato juntos,...","Que sean materialistas, detallistas y escuchen.","Hablar juntos, ya sea comiendo, corriendo, jug...",Simplemente estar con esa persona lo hace idea...,"Something About You de Eyedress y Dent May, re..."
4,31/10/2026 13:10:25,424000455@pcpuma.acatlan.unam.mx,Sí,Área 1: Ciencias Físico-Matemáticas y de las I...,MAC,7.0,No,12.0,Que te atraiga la persona y que la aceptes com...,Honestidad,Que respete mis límites,Empatía,Alto,Que sea mentiroso que sea indeciso,Controlador que sea grosero,2 años más que yo,Detalles tiempo de calidad,Sorprenderme en mi cumpleaños,Hablarlo,Una cita romántica,Love Rosie


## 2. Exploración inicial

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 21 columns):
 #   Column                                                                                Non-Null Count  Dtype  
---  ------                                                                                --------------  -----  
 0   Marca temporal                                                                        20 non-null     str    
 1   Dirección de correo electrónico                                                       20 non-null     str    
 2   ¿Deseas participar en el estudio?                                                     20 non-null     str    
 3   ¿A qué área de conocimiento pertenece tu carrera?                                     19 non-null     str    
 4   ¿Cuál es tu carrera?                                                                  19 non-null     str    
 5   ¿En qué semestre te encuentras actualmente?                                           19 non-null 

In [15]:
# Seleccionamos la columna de quién desea participar y la de tiempo
col_participar = df.columns[2]
col_tiempo = df.columns[0]

# Mapear "Sí" → 1, "No" → 0
df[col_participar] = df[col_participar].map({"Sí": 1, "No": 0})
df = df[df[col_participar] == 1].copy()

# Eliminar columnas innecesarias
df = df.drop(columns=[col_participar, col_tiempo])

In [16]:
# Extraer números de cuenta
df = df.rename(columns={df.columns[0]: "No. Cuenta"})
col_cuenta = df.columns[0]
df[col_cuenta] = df[col_cuenta].astype(str).str.extract(r'(\d{9})')

df.head()

,No. Cuenta,¿A qué área de conocimiento pertenece tu carrera?,¿Cuál es tu carrera?,¿En qué semestre te encuentras actualmente?,¿Tienes pareja actualmente?,¿A qué edad fue tu primera relación amorosa?,¿Qué significa para ti estar enamorado?,¿Cuál consideras que es la base de la confianza?,¿Cómo defines el respeto dentro de una relación?,¿Qué cualidad valoras más en una pareja y por qué?,Describe físicamente a tu pareja ideal,"¿Cuáles son tus ""no negociables"" en una relación?",¿Cuáles son las red flags (señales de alerta) que consideras dentro de una relación?,¿Cuál es la diferencia de edad máxima que aceptarías en una pareja?,¿Cómo demuestras cariño sin usar palabras?,¿Qué es lo más significativo que alguien ha hecho por ti?,¿Cuál sería para ti la forma correcta de resolver un desacuerdo con tu pareja?,"Describe brevemente una cita ideal, sin importar presupuesto ni lugar.","¿Qué canción, película o libro describe mejor tu idea del amor?"
1,321188065,Área 1: Ciencias Físico-Matemáticas y de las I...,Matemáticas Aplicadas y Computación,7.0,Si,14.0,Es un sentimiento dirigido hacia otra persona ...,Honestidad y confianza,No lastimar física o emocionalmente a la otra ...,"El apoyo, porque me hace sentir especial e imp...",No me fijo físicamente en alguien.,"Infidelidad, mentiras, engaños y tratar de con...",Frases para controlar o atacar emocionalmente ...,+/- 2 años,Querer estar cerca de la otra persona.\nDarle ...,Tratarme cómo una persona normal y ser paciente.,Hablar de lo que pasó cuando el enojo haya pas...,Un viaje a otro país.,Cómo si fuera la primera vez
2,424059594,Área 1: Ciencias Físico-Matemáticas y de las I...,Matemáticas Aplicadas y Computación,7.0,No,12.0,Que una persona te genere muchas emociones y n...,Honestidad y lealtad,Ser leal,Cariño ya que me hace sentir querido por esa p...,"Leal, respetuosa, cariñosa y con buen sentido ...",Infidelidad y Falta de Respeto,Que me hable mucho de sus relaciones anteriore...,+/- 4 años,"Tiempo, detalles, palabras y sobre todo estar ...",Escucharme hablar del mismo tema infinidad de ...,Hablando y teniendo una buena comunicación,Un día en un parque de diversiones y después u...,La canción Mi Riqueza
3,423085822,Área 1: Ciencias Físico-Matemáticas y de las I...,MAC,7.0,Si,15.0,Escuche Amar y querer de José José para más de...,"Comunicación, la capacidad de ser vulnerable.","No pasarse de ""chistoso"".","Comunicación, porque así puedo estar en sus za...",De baja estatura.,-Infidelidad. \n-Manipulación.\n-Falta de empa...,"El que violente física, emocionalmente de mi. ...",+/- 3 años,"Tener retroalimentación, pasar un rato juntos,...","Que sean materialistas, detallistas y escuchen.","Hablar juntos, ya sea comiendo, corriendo, jug...",Simplemente estar con esa persona lo hace idea...,"Something About You de Eyedress y Dent May, re..."
4,424000455,Área 1: Ciencias Físico-Matemáticas y de las I...,MAC,7.0,No,12.0,Que te atraiga la persona y que la aceptes com...,Honestidad,Que respete mis límites,Empatía,Alto,Que sea mentiroso que sea indeciso,Controlador que sea grosero,2 años más que yo,Detalles tiempo de calidad,Sorprenderme en mi cumpleaños,Hablarlo,Una cita romántica,Love Rosie
5,424144407,Área 1: Ciencias Físico-Matemáticas y de las I...,Matemáticas aplicadas y Computación,7.0,Si,13.0,Para mí estar enamorado es querer compartir tu...,"Comunicación, respeto y sinceridad",Simplemente tener tratos buenos ( exclusividad...,"El ser escuchado, la atención a los detalles m...",Basándome en mi pareja y patrones que he tenid...,Infidelidad y no tener tiempo de calidad,"Revisar el teléfono, que le “caigan mal” mis a...",-2/+4 años de diferencia,"Tiempo de calidad, pequeños detalles, poder se...","Demostrarme que no estoy sola, sin quitarme mi...",Hablando las veces y tiempo que sea necesario ...,Un día de pícnic,Off Campus


In [17]:
display(df.describe(include='all'))

,No. Cuenta,¿A qué área de conocimiento pertenece tu carrera?,¿Cuál es tu carrera?,¿En qué semestre te encuentras actualmente?,¿Tienes pareja actualmente?,¿A qué edad fue tu primera relación amorosa?,¿Qué significa para ti estar enamorado?,¿Cuál consideras que es la base de la confianza?,¿Cómo defines el respeto dentro de una relación?,¿Qué cualidad valoras más en una pareja y por qué?,Describe físicamente a tu pareja ideal,"¿Cuáles son tus ""no negociables"" en una relación?",¿Cuáles son las red flags (señales de alerta) que consideras dentro de una relación?,¿Cuál es la diferencia de edad máxima que aceptarías en una pareja?,¿Cómo demuestras cariño sin usar palabras?,¿Qué es lo más significativo que alguien ha hecho por ti?,¿Cuál sería para ti la forma correcta de resolver un desacuerdo con tu pareja?,"Describe brevemente una cita ideal, sin importar presupuesto ni lugar.","¿Qué canción, película o libro describe mejor tu idea del amor?"
count,19,19,19,19.0,19,19.000000,19,19,19,19,19,19,19,19,19,19,19,19,19
unique,19,1,11,NaN,4,NaN,19,19,19,19,19,19,19,16,19,19,18,19,19
top,321188065,Área 1: Ciencias Físico-Matemáticas y de las I...,Matemáticas Aplicadas y Computación,NaN,No,NaN,Es un sentimiento dirigido hacia otra persona ...,Honestidad y confianza,No lastimar física o emocionalmente a la otra ...,"El apoyo, porque me hace sentir especial e imp...",No me fijo físicamente en alguien.,"Infidelidad, mentiras, engaños y tratar de con...",Frases para controlar o atacar emocionalmente ...,3 años de diferencia,Querer estar cerca de la otra persona.\nDarle ...,Tratarme cómo una persona normal y ser paciente.,Hablarlo,Un viaje a otro país.,Cómo si fuera la primera vez
freq,1,19,4,NaN,9,NaN,1,1,1,1,1,1,1,2,1,1,2,1,1
mean,NaN,NaN,NaN,7.0,NaN,12.684211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.0,NaN,6.280947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,7.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,7.0,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,7.0,NaN,14.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,7.0,NaN,16.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Eliminando la columna del área de conocimiento (todos son de la 1)
# Eliminando la columna del semestre, todos son de 7mo
df = df.drop(columns=[df.columns[1], df.columns[3]])

,No. Cuenta,¿Cuál es tu carrera?,¿Tienes pareja actualmente?,¿A qué edad fue tu primera relación amorosa?,¿Qué significa para ti estar enamorado?,¿Cuál consideras que es la base de la confianza?,¿Cómo defines el respeto dentro de una relación?,¿Qué cualidad valoras más en una pareja y por qué?,Describe físicamente a tu pareja ideal,"¿Cuáles son tus ""no negociables"" en una relación?",¿Cuáles son las red flags (señales de alerta) que consideras dentro de una relación?,¿Cuál es la diferencia de edad máxima que aceptarías en una pareja?,¿Cómo demuestras cariño sin usar palabras?,¿Qué es lo más significativo que alguien ha hecho por ti?,¿Cuál sería para ti la forma correcta de resolver un desacuerdo con tu pareja?,"Describe brevemente una cita ideal, sin importar presupuesto ni lugar.","¿Qué canción, película o libro describe mejor tu idea del amor?"
1,321188065,Matemáticas Aplicadas y Computación,Si,14.0,Es un sentimiento dirigido hacia otra persona ...,Honestidad y confianza,No lastimar física o emocionalmente a la otra ...,"El apoyo, porque me hace sentir especial e imp...",No me fijo físicamente en alguien.,"Infidelidad, mentiras, engaños y tratar de con...",Frases para controlar o atacar emocionalmente ...,+/- 2 años,Querer estar cerca de la otra persona.\nDarle ...,Tratarme cómo una persona normal y ser paciente.,Hablar de lo que pasó cuando el enojo haya pas...,Un viaje a otro país.,Cómo si fuera la primera vez
2,424059594,Matemáticas Aplicadas y Computación,No,12.0,Que una persona te genere muchas emociones y n...,Honestidad y lealtad,Ser leal,Cariño ya que me hace sentir querido por esa p...,"Leal, respetuosa, cariñosa y con buen sentido ...",Infidelidad y Falta de Respeto,Que me hable mucho de sus relaciones anteriore...,+/- 4 años,"Tiempo, detalles, palabras y sobre todo estar ...",Escucharme hablar del mismo tema infinidad de ...,Hablando y teniendo una buena comunicación,Un día en un parque de diversiones y después u...,La canción Mi Riqueza
3,423085822,MAC,Si,15.0,Escuche Amar y querer de José José para más de...,"Comunicación, la capacidad de ser vulnerable.","No pasarse de ""chistoso"".","Comunicación, porque así puedo estar en sus za...",De baja estatura.,-Infidelidad. \n-Manipulación.\n-Falta de empa...,"El que violente física, emocionalmente de mi. ...",+/- 3 años,"Tener retroalimentación, pasar un rato juntos,...","Que sean materialistas, detallistas y escuchen.","Hablar juntos, ya sea comiendo, corriendo, jug...",Simplemente estar con esa persona lo hace idea...,"Something About You de Eyedress y Dent May, re..."
4,424000455,MAC,No,12.0,Que te atraiga la persona y que la aceptes com...,Honestidad,Que respete mis límites,Empatía,Alto,Que sea mentiroso que sea indeciso,Controlador que sea grosero,2 años más que yo,Detalles tiempo de calidad,Sorprenderme en mi cumpleaños,Hablarlo,Una cita romántica,Love Rosie
5,424144407,Matemáticas aplicadas y Computación,Si,13.0,Para mí estar enamorado es querer compartir tu...,"Comunicación, respeto y sinceridad",Simplemente tener tratos buenos ( exclusividad...,"El ser escuchado, la atención a los detalles m...",Basándome en mi pareja y patrones que he tenid...,Infidelidad y no tener tiempo de calidad,"Revisar el teléfono, que le “caigan mal” mis a...",-2/+4 años de diferencia,"Tiempo de calidad, pequeños detalles, poder se...","Demostrarme que no estoy sola, sin quitarme mi...",Hablando las veces y tiempo que sea necesario ...,Un día de pícnic,Off Campus


## 5. Exportar datos limpios

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_CSV, index=False)
print(f"✓ Datos limpios guardados en: {PROCESSED_CSV}")
print(f"  Filas: {len(df)}")
print(f"  Columnas: {len(df.columns)}")